In [1]:
!pip install git+https://github.com/haoheliu/voicefixer.git pedalboard soundfile librosa

  Cloning https://github.com/haoheliu/voicefixer.git to /tmp/pip-req-build-7qf38gdu
  Running command git clone --filter=blob:none --quiet https://github.com/haoheliu/voicefixer.git /tmp/pip-req-build-7qf38gdu
  Resolved https://github.com/haoheliu/voicefixer.git to commit aae2253c85f97a87b844b6832384de249c23ab37
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.5/58.5 kB 4.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 101.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 106.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 78.7 MB/s eta 0:00:00
  Created wheel for voicefixer: filename=voicefixer-0.1.2-py3-none-any.whl size=51867 sha256=8f5a93f39cd20c16b6ee49705a3fc95884cd08745377e5881cc24da78a03997b
  Stored in directory: /tmp/pip-ephem-wheel-cache-54gl0itj/wheels/14/b9/4b/dfefbadcd7953c01e4b380e8b987d6cc536ef5631dbfafdd88
  Cr

In [3]:
!sudo apt-get update && sudo apt-get install -y ffmpeg

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [4]:
import os
import torch
import librosa
import soundfile as sf
from voicefixer import VoiceFixer
from pedalboard import Pedalboard, HighpassFilter, PeakFilter, Compressor, Gain

# Check if GPU acceleration is available for the AI engine
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using processing unit: {device.upper()}")

print("Initializing VoiceFixer AI Engine...")
vf = VoiceFixer()
print("AI Engine ready.")

Weights downloaded in: /root/.cache/voicefixer/synthesis_module/44100/model.ckpt-1490000_trimed.pt Size: 135613039
Weights downloaded in: /root/.cache/voicefixer/analysis_module/checkpoints/vf.ckpt Size: 489307071
Using processing unit: CUDA
Initializing VoiceFixer AI Engine...
AI Engine ready.


In [5]:
# 1. Define the file path variables explicitly in this block
input_mpeg_path = "/content/Leeroy Jenkins HD 1080p - J Jonah Jameson (youtube).wav"
converted_wav_path = "converted_input.wav"
ai_restored_path = "ai_restored_output.wav"

print(f"Step 1A: Converting {input_mpeg_path} to standardized PCM WAV...")
try:
    audio_signal, sample_rate = librosa.load(input_mpeg_path, sr=44100)
    sf.write(converted_wav_path, audio_signal, sample_rate)
    print("MPEG conversion successful.")
except Exception as e:
    print(f"Error converting your MPEG file: {e}")

print("\nStep 1B: Running AI Speech Restoration...")
vf.restore(
    input=converted_wav_path,
    output=ai_restored_path,
    cuda=torch.cuda.is_available(),
    mode=0  # If speech sounds a little muffled, you can try changing this to 1 later
)

print(f"AI Restoration complete. Saved to variable variable path: {ai_restored_path}")

Step 1A: Converting /content/Leeroy Jenkins HD 1080p - J Jonah Jameson (youtube).wav to standardized PCM WAV...
MPEG conversion successful.

Step 1B: Running AI Speech Restoration...
AI Restoration complete. Saved to variable variable path: ai_restored_output.wav


In [6]:
final_output_path = "crystal_clear_speech.wav"

print("Step 2A: Configuring DSP Mastering Chain...")
dsp_board = Pedalboard([
    HighpassFilter(cutoff_frequency_hz=85.0),
    PeakFilter(cutoff_frequency_hz=6000.0, gain_db=6.0, q=0.7), # Adds crisp sharpness
    Compressor(threshold_db=-14.0, ratio=3.0, attack_ms=10.0, release_ms=100.0),
    Gain(gain_db=2.0)
])

print("Step 2B: Rendering final intelligible audio file...")
# This will now successfully find 'ai_restored_path' because Step 2 was run
audio_data, sample_rate = librosa.load(ai_restored_path, sr=None)
mastered_audio = dsp_board(audio_data, sample_rate)
sf.write(final_output_path, mastered_audio, sample_rate)

print("\n--- PROCESS COMPLETED ---")
print(f"Your final intelligible audio file is ready at: {final_output_path}")

Step 2A: Configuring DSP Mastering Chain...
Step 2B: Rendering final intelligible audio file...

--- PROCESS COMPLETED ---
Your final intelligible audio file is ready at: crystal_clear_speech.wav


In [7]:
# ============================================================
# PASTE THIS AS A NEW CELL AT THE END OF YOUR EXISTING COLAB NOTEBOOK
# (after the cells that define vf, converted_wav_path, ai_restored_path, etc.)
# ============================================================

!pip install pyloudnorm -q

import time
import numpy as np
import librosa
import soundfile as sf
import pyloudnorm as pyln

# ------------------------------------------------------------
# 1. LATENCY — time each stage you already ran above
# ------------------------------------------------------------
# Re-run the same 3 stages here, wrapped in timers, so we get real numbers.

timings = {}

t0 = time.perf_counter()
audio_signal, sample_rate = librosa.load(input_mpeg_path, sr=44100)
sf.write(converted_wav_path, audio_signal, sample_rate)
timings["convert_to_wav"] = time.perf_counter() - t0

t0 = time.perf_counter()
vf.restore(
    input=converted_wav_path,
    output=ai_restored_path,
    cuda=torch.cuda.is_available(),
    mode=0
)
timings["voicefixer_restore"] = time.perf_counter() - t0

t0 = time.perf_counter()
dsp_board = Pedalboard([
    HighpassFilter(cutoff_frequency_hz=85.0),
    PeakFilter(cutoff_frequency_hz=6000.0, gain_db=6.0, q=0.7),
    Compressor(threshold_db=-14.0, ratio=3.0, attack_ms=10.0, release_ms=100.0),
    Gain(gain_db=2.0)
])
audio_data, sr = librosa.load(ai_restored_path, sr=None)
mastered_audio = dsp_board(audio_data, sr)
sf.write(final_output_path, mastered_audio, sr)
timings["pedalboard_mastering"] = time.perf_counter() - t0

timings["TOTAL"] = sum(timings.values())

print("=== LATENCY (seconds) ===")
for stage, secs in timings.items():
    print(f"{stage:<24s} {secs:8.3f}s")

# Also worth reporting as "real-time factor": how many seconds of
# processing per second of audio -- more meaningful than raw seconds.
audio_duration_sec = librosa.get_duration(y=audio_signal, sr=sample_rate)
rtf = timings["TOTAL"] / audio_duration_sec
print(f"\nInput audio duration: {audio_duration_sec:.1f}s")
print(f"Real-time factor: {rtf:.2f}x "
      f"({'faster than real-time' if rtf < 1 else 'slower than real-time'})")


# ------------------------------------------------------------
# 2. NOISE FLOOR REDUCTION (dB) — converted (before) vs final (after)
# ------------------------------------------------------------
def estimate_noise_floor_db(waveform, sr, percentile=10):
    frame_len = int(0.025 * sr)
    hop_len = int(0.010 * sr)
    frames = librosa.util.frame(waveform, frame_length=frame_len, hop_length=hop_len)
    rms_per_frame = np.sqrt(np.mean(frames ** 2, axis=0) + 1e-12)
    noise_floor_rms = np.percentile(rms_per_frame, percentile)
    return 20 * np.log10(noise_floor_rms + 1e-12)

before, sr_b = librosa.load(converted_wav_path, sr=None)
after, sr_a = librosa.load(final_output_path, sr=None)

noise_before = estimate_noise_floor_db(before, sr_b)
noise_after = estimate_noise_floor_db(after, sr_a)

print(f"\n=== NOISE FLOOR ===")
print(f"Before: {noise_before:6.2f} dB")
print(f"After:  {noise_after:6.2f} dB")
print(f"Reduction: {noise_before - noise_after:6.2f} dB")


# ------------------------------------------------------------
# 3. LOUDNESS CHANGE (LUFS) — shows what Pedalboard's mastering did
# ------------------------------------------------------------
before_sf, sr_before = sf.read(converted_wav_path)
after_sf, sr_after = sf.read(final_output_path)

meter_before = pyln.Meter(sr_before)
meter_after = pyln.Meter(sr_after)

loudness_before = meter_before.integrated_loudness(before_sf)
loudness_after = meter_after.integrated_loudness(after_sf)

print(f"\n=== LOUDNESS (LUFS) ===")
print(f"Before: {loudness_before:6.2f} LUFS")
print(f"After:  {loudness_after:6.2f} LUFS")
print(f"Change: {loudness_after - loudness_before:+6.2f} LU")

=== LATENCY (seconds) ===
convert_to_wav              0.451s
voicefixer_restore          9.698s
pedalboard_mastering        0.295s
TOTAL                      10.444s

Input audio duration: 171.2s
Real-time factor: 0.06x (faster than real-time)

=== NOISE FLOOR ===
Before: -32.72 dB
After:  -70.27 dB
Reduction:  37.55 dB

=== LOUDNESS (LUFS) ===
Before:  -5.53 LUFS
After:  -15.54 LUFS
Change: -10.01 LU
